In [1]:
from pathlib import Path
import json
import pickle 
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.nn.utils.rnn import pad_sequence
import gc
from tqdm.auto import tqdm
import tempfile
import uuid
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import matplotlib.pyplot as plt
import koreanize_matplotlib
import re

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)
print(PROJECT_ROOT / 'src' / 'data' / 'dataset.py')
print((PROJECT_ROOT / 'src' / 'data' / 'dataset.py').exists())

c:\Users\aneug\OneDrive\Desktop\FinalProject
c:\Users\aneug\OneDrive\Desktop\FinalProject\src\data\dataset.py
True


In [4]:
from src.data.dataset import SessionDataset, collate_fn

from src.data.preprocess import (
    normalize_empty_string,
    parse_sessions
)

from src.data.pipeline import (
    split_sessions,
    encode_session_locales,
    process_catalog,
    split_xy
)

from src.models.sasrec import SemanticSASRec
from src.models.gru import SemanticGRURec

from src.training import (
    calculate_logits,
    train_model
)

from src.evaluation.full_catalog import evaluate_full_catalog

from src.inference import (
    recommend_next_items,
    get_product_metadata
)

ImportError: cannot import name 'SessionDataset' from 'src.data.dataset' (c:\Users\aneug\OneDrive\Desktop\FinalProject\src\data\dataset.py)

In [5]:
import src.data.dataset as dataset_module

print(dataset_module.__file__)
print(hasattr(dataset_module, 'SessionDataset'))
print(hasattr(dataset_module, 'collate_fn'))

c:\Users\aneug\OneDrive\Desktop\FinalProject\src\data\dataset.py
False
False


In [ ]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = (
    PROJECT_ROOT
    / 'artifacts'
    / 'embeddings'
    / 'voyage4_large_1024'
)
MAPPINGS_DIR = PROJECT_ROOT / 'artifacts' / 'mappings'

In [ ]:
train = pd.read_parquet(
    PROCESSED_DIR / 'train.parquet'
)

val = pd.read_parquet(
    PROCESSED_DIR / 'val.parquet'
)

test = pd.read_parquet(
    PROCESSED_DIR / 'test.parquet'
)

num_catalog = pd.read_parquet(
    PROCESSED_DIR / 'catalog_numeric.parquet'
)

locale_embeddings = np.load(
    EMBEDDINGS_DIR / 'product_embeddings.npy',
    mmap_mode='r'
)

embedding_num_ids = np.load(
    EMBEDDINGS_DIR / 'embedding_num_ids.npy'
)

embedding_metadata = pd.read_pickle(
    EMBEDDINGS_DIR / 'embedding_metadata.pkl'
)

with open(
    EMBEDDINGS_DIR / 'config.json',
    'r',
    encoding='utf-8'
) as f:
    embedding_config = json.load(f)
    
with open(
    MAPPINGS_DIR / 'items_dict.pkl',
    'rb'
) as f:
    items_dict = pickle.load(f)


In [ ]:
print(train.shape)
print(val.shape)
print(test.shape)
print(locale_embeddings.shape)
print(embedding_num_ids.shape)
print(len(items_dict))

In [ ]:
class SessionDataset(Dataset):
    def __init__(self, X, y):
        sequence_columns = [
            col for col in X.columns
            if 'remainder__prev_items_list' in col
        ]

        if len(sequence_columns) != 1:
            raise ValueError(
                f'Expected one sequence column, found {sequence_columns}'
            )

        self.sequence_col = sequence_columns[0]

        self.locale_cols = [
            col for col in X.columns
            if col != self.sequence_col
        ]

        self.sequences = X[self.sequence_col].tolist()

        self.locales = X[
            self.locale_cols
        ].to_numpy(dtype=np.float32)

        self.targets = y.to_numpy(
            dtype=np.int64
        )

    def __len__(self):
        return len(self.sequences)
    
    
    def __getitem__(self, idx):
        sequence = torch.tensor(
            self.sequences[idx],
            dtype=torch.long
        )

        locale = torch.tensor(
            self.locales[idx],
            dtype=torch.float32
        )

        target = torch.tensor(
            self.targets[idx],
            dtype=torch.long
        )

        return sequence, locale, target

In [ ]:
def collate_fn(batch, padding_value = 0):
    sequences, locales, targets = zip(*batch)
    
    sequences_padded = pad_sequence(
        sequences, 
        batch_first = True, 
        padding_value = padding_value
    ) 
    
    locales_stacked = torch.stack(locales, dim=0)
    targets_stacked = torch.stack(targets, dim=0)
    return sequences_padded, locales_stacked, targets_stacked
    

In [ ]:
train_dataset = SessionDataset(
    train.drop(columns = 'target'), 
    train['target']
)

val_dataset = SessionDataset(
    val.drop(columns = 'target'), 
    val['target']
)

test_dataset = SessionDataset(
    test.drop(columns = 'target'), 
    test['target']
)

In [ ]:
SOURCE_PATH = EMBEDDINGS_DIR / 'product_embeddings.npy'
IDS_PATH = EMBEDDINGS_DIR / 'embedding_num_ids.npy'

GLOBAL_PATH = EMBEDDINGS_DIR / 'global_product_embeddings.npy'
COUNTS_PATH = EMBEDDINGS_DIR / 'global_embedding_representation_counts.npy'

temp_dir = Path(tempfile.gettempdir())
SUMS_PATH = temp_dir / f'global_embedding_sums_{uuid.uuid4().hex}.npy'


def close_memmap(array):
    if isinstance(array, np.memmap):
        array.flush()
        if getattr(array, '_mmap', None) is not None:
            array._mmap.close()


locale_embeddings = np.load(
    SOURCE_PATH,
    mmap_mode='r'
)

embedding_num_ids = np.load(
    IDS_PATH,
    mmap_mode='r'
)

N_ITEMS = int(embedding_num_ids.max())
EMBEDDING_DIM = int(locale_embeddings.shape[1])

assert locale_embeddings.shape[0] == len(embedding_num_ids)

print(f'지역별 임베딩 수 | Locale representations: {len(embedding_num_ids):,}')
print(f'상품 ID 범위 | Item ID range: 1 ... {N_ITEMS:,}')
print(f'새 임베딩 행렬 크기 | Global matrix shape: {(N_ITEMS + 1, EMBEDDING_DIM)}')
print(f'임시 파일 경로 | Temporary file: {SUMS_PATH}')
print('---' * 35)


embedding_sums = np.lib.format.open_memmap(
    SUMS_PATH,
    mode='w+',
    dtype=np.float32,
    shape=(N_ITEMS + 1, EMBEDDING_DIM)
)

embedding_sums[:] = 0.0

representation_counts = np.zeros(
    N_ITEMS + 1,
    dtype=np.int32
)

SOURCE_BATCH_SIZE = 4096

for start in tqdm(
    range(0, len(embedding_num_ids), SOURCE_BATCH_SIZE),
    desc='1/2 Aggregating locale embeddings'
):
    end = min(
        start + SOURCE_BATCH_SIZE,
        len(embedding_num_ids)
    )

    batch_ids = np.asarray(
        embedding_num_ids[start:end],
        dtype=np.int64
    )

    batch_embeddings = np.asarray(
        locale_embeddings[start:end],
        dtype=np.float32
    )

    np.add.at(
        embedding_sums,
        batch_ids,
        batch_embeddings
    )

    np.add.at(
        representation_counts,
        batch_ids,
        1
    )

embedding_sums.flush()

missing_mask = representation_counts[1:] == 0

if missing_mask.any():
    missing_ids = np.where(missing_mask)[0] + 1

    close_memmap(embedding_sums)
    del embedding_sums
    gc.collect()

    raise ValueError(
        f'Items without semantic embeddings: {len(missing_ids):,}'
    )


if GLOBAL_PATH.exists():
    GLOBAL_PATH.unlink()

global_embeddings = np.lib.format.open_memmap(
    GLOBAL_PATH,
    mode='w+',
    dtype=np.float16,
    shape=(N_ITEMS + 1, EMBEDDING_DIM)
)

global_embeddings[0] = np.zeros(
    EMBEDDING_DIM,
    dtype=np.float16
)

ITEM_BATCH_SIZE = 4096

for start in tqdm(
    range(1, N_ITEMS + 1, ITEM_BATCH_SIZE),
    desc='2/2 Calculating mean embeddings'
):
    end = min(
        start + ITEM_BATCH_SIZE,
        N_ITEMS + 1
    )

    batch_sums = np.asarray(
        embedding_sums[start:end],
        dtype=np.float32
    )

    batch_counts = (
        representation_counts[start:end]
        .astype(np.float32)
        .reshape(-1, 1)
    )

    batch_means = (
        batch_sums / batch_counts
    )

    global_embeddings[start:end] = (
        batch_means.astype(np.float16)
    )

global_embeddings.flush()

np.save(
    COUNTS_PATH,
    representation_counts
)

close_memmap(global_embeddings)
close_memmap(embedding_sums)
close_memmap(locale_embeddings)
close_memmap(embedding_num_ids)

del global_embeddings
del embedding_sums
del locale_embeddings
del embedding_num_ids

gc.collect()

try:
    SUMS_PATH.unlink()
    print('임시 파일 삭제 완료 | Temporary aggregation file deleted')
except PermissionError:
    print(
        f'임시 파일을 삭제할 수 없습니다 | '
        f'Could not delete temporary file: {SUMS_PATH}'
    )

global_embeddings = np.load(
    GLOBAL_PATH,
    mmap_mode='r'
)

representation_counts = np.load(
    COUNTS_PATH
)

assert global_embeddings.shape == (
    N_ITEMS + 1,
    EMBEDDING_DIM
)

assert np.all(
    global_embeddings[0] == 0
)

assert np.isfinite(
    global_embeddings[:1000]
).all()

assert np.isfinite(
    global_embeddings[-1000:]
).all()

assert representation_counts[0] == 0

assert representation_counts[1:].min() >= 1

print()
print('=' * 100)
print('전역 상품 임베딩 생성 완료 | GLOBAL PRODUCT EMBEDDINGS CREATED')
print('=' * 100)
print(f'Shape: {global_embeddings.shape}')
print(f'Dtype: {global_embeddings.dtype}')
print(f'PAD vector norm: {np.linalg.norm(global_embeddings[0]):.4f}')
print(f'Min representations per item: {representation_counts[1:].min()}')
print(f'Max representations per item: {representation_counts[1:].max()}')
print(f'Mean representations per item: {representation_counts[1:].mean():.4f}')
print(f'Global embeddings saved to: {GLOBAL_PATH}')
print(f'Representation counts saved to: {COUNTS_PATH}')
print('=' * 100)


In [ ]:
MAX_SEQ_LEN = 64
BATCH_SIZE = 128

def collate_fn(batch, padding_value=0):
    sequences, locales, targets = zip(*batch)

    sequences = [sequence[-MAX_SEQ_LEN:] for sequence in sequences]

    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=padding_value)
    locales_stacked = torch.stack(locales, dim=0)
    targets_stacked = torch.stack(targets, dim=0)

    return sequences_padded, locales_stacked, targets_stacked


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=torch.cuda.is_available())

In [ ]:
GLOBAL_EMBEDDINGS_PATH = EMBEDDINGS_DIR / 'global_product_embeddings.npy'

global_embeddings = np.load(GLOBAL_EMBEDDINGS_PATH, mmap_mode='r')

N_ITEMS = global_embeddings.shape[0] - 1
SEMANTIC_DIM = global_embeddings.shape[1]

print(global_embeddings.shape)
print(global_embeddings.dtype)
print(f'Number of items: {N_ITEMS:,}')

In [ ]:
class SemanticSASRec(nn.Module):
    def __init__(self, semantic_dim=1024, d_model=128, locale_dim=6, max_seq_len=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()

        self.semantic_head = nn.Sequential(
            nn.Linear(semantic_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.locale_head = nn.Linear(locale_dim, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_norm = nn.LayerNorm(d_model)

    def project_items(self, semantic_vectors):
        item_vectors = self.semantic_head(semantic_vectors)
        return F.normalize(item_vectors, dim=-1)

    def forward(self, semantic_sequences, locales, padding_mask):
        batch_size, seq_len, _ = semantic_sequences.shape

        x = self.semantic_head(semantic_sequences)

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        position_vectors = self.position_embedding(positions)
        locale_vectors = self.locale_head(locales).unsqueeze(1)

        x = x + position_vectors + locale_vectors
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1)

        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=padding_mask)

        lengths = (~padding_mask).sum(dim=1)
        last_indexes = lengths - 1
        batch_indexes = torch.arange(batch_size, device=x.device)

        session_vectors = x[batch_indexes, last_indexes]
        session_vectors = self.output_norm(session_vectors)

        return F.normalize(session_vectors, dim=-1)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SemanticSASRec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

print(DEVICE)
print(model)

In [ ]:
def semantic_lookup(item_ids):
    ids_numpy = item_ids.detach().cpu().numpy()
    vectors = np.asarray(global_embeddings[ids_numpy], dtype=np.float32)
    return torch.from_numpy(vectors)

In [ ]:
NUM_NEGATIVES = 128

def sample_negatives(targets, num_items, num_negatives):
    targets_cpu = targets.detach().cpu()

    negatives = torch.randint(
        low=1,
        high=num_items + 1,
        size=(len(targets_cpu), num_negatives)
    )

    target_matrix = targets_cpu.unsqueeze(1)

    mask = negatives.eq(target_matrix)

    while mask.any():
        negatives[mask] = torch.randint(
            low=1,
            high=num_items + 1,
            size=(mask.sum().item(),)
        )

        mask = negatives.eq(target_matrix)

    return negatives

In [ ]:
TEMPERATURE = 0.07

def calculate_logits(model, sequences, locales, targets):
    padding_mask = sequences.eq(0)

    semantic_sequences = semantic_lookup(sequences).to(DEVICE, non_blocking=True)
    target_semantics = semantic_lookup(targets).to(DEVICE, non_blocking=True)

    negative_ids = sample_negatives(targets, N_ITEMS, NUM_NEGATIVES)
    negative_semantics = semantic_lookup(negative_ids).to(DEVICE, non_blocking=True)

    sequences = sequences.to(DEVICE, non_blocking=True)
    locales = locales.to(DEVICE, non_blocking=True)
    padding_mask = padding_mask.to(DEVICE, non_blocking=True)

    session_vectors = model(semantic_sequences, locales, padding_mask)

    positive_vectors = model.project_items(target_semantics)
    negative_vectors = model.project_items(negative_semantics)

    positive_scores = (session_vectors * positive_vectors).sum(dim=1, keepdim=True)
    negative_scores = torch.einsum('bd,bkd->bk', session_vectors, negative_vectors)

    logits = torch.cat([positive_scores, negative_scores], dim=1)
    logits = logits / TEMPERATURE

    labels = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)

    return logits, labels

In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

RUN_DIR = PROJECT_ROOT / 'runs' / 'semantic_sasrec_baseline'
CHECKPOINT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(RUN_DIR))

In [ ]:
VAL_MAX_BATCHES = 500

def validate(model, val_loader):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    recall_10_sum = 0.0
    ndcg_10_sum = 0.0

    with torch.no_grad():
        progress = tqdm(val_loader, total=min(len(val_loader), VAL_MAX_BATCHES), desc='Validation', leave=False)

        for batch_idx, (sequences, locales, targets) in enumerate(progress):
            if batch_idx >= VAL_MAX_BATCHES:
                break

            logits, labels = calculate_logits(model, sequences, locales, targets)
            loss = criterion(logits, labels)

            batch_size = logits.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            ranking = torch.argsort(logits, dim=1, descending=True)
            positive_ranks = (ranking == 0).nonzero(as_tuple=False)[:, 1] + 1

            recall_10_sum += (positive_ranks <= 10).float().sum().item()

            ndcg = torch.where(
                positive_ranks <= 10,
                1.0 / torch.log2(positive_ranks.float() + 1.0),
                torch.zeros_like(positive_ranks, dtype=torch.float32)
            )

            ndcg_10_sum += ndcg.sum().item()

    val_loss = total_loss / total_samples
    recall_10 = recall_10_sum / total_samples
    ndcg_10 = ndcg_10_sum / total_samples

    return val_loss, recall_10, ndcg_10

In [ ]:
best_val_loss = float('inf')
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, (sequences, locales, targets) in enumerate(progress, start=1):
        optimizer.zero_grad(set_to_none=True)

        logits, labels = calculate_logits(model, sequences, locales, targets)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()
        global_step += 1

        writer.add_scalar('Train/Loss_step', loss.item(), global_step)

        progress.set_postfix(
            loss=f'{loss.item():.4f}',
            avg=f'{running_loss / step:.4f}'
        )

    train_loss = running_loss / len(train_loader)

    val_loss, recall_10, ndcg_10 = validate(model, val_loader)

    writer.add_scalar('Train/Loss_epoch', train_loss, epoch)
    writer.add_scalar('Validation/Loss', val_loss, epoch)
    writer.add_scalar('Validation/Sampled_Recall@10', recall_10, epoch)
    writer.add_scalar('Validation/Sampled_NDCG@10', ndcg_10, epoch)

    print()
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'Train loss: {train_loss:.4f}')
    print(f'Validation loss: {val_loss:.4f}')
    print(f'Sampled Recall@10: {recall_10:.4f}')
    print(f'Sampled NDCG@10: {ndcg_10:.4f}')
    print()

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'recall_10': recall_10,
        'ndcg_10': ndcg_10
    }

    torch.save(checkpoint, CHECKPOINT_DIR / 'semantic_sasrec_last.pt')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, CHECKPOINT_DIR / 'semantic_sasrec_best.pt')
        print('Best checkpoint saved.')

writer.close()

print('Training completed.')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../runs

In [ ]:
EVALUATION_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'semantic_sasrec_best.pt'
GLOBAL_EMBEDDINGS_PATH = EMBEDDINGS_DIR / 'global_product_embeddings.npy'
METRICS_PATH = EVALUATION_DIR / 'semantic_sasrec_full_catalog_metrics.json'

EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20000
TOP_K = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if DEVICE.type != 'cuda':
    raise RuntimeError('Full-catalog evaluation should be run on GPU.')

gc.collect()
torch.cuda.empty_cache()

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

global_embeddings = np.load(GLOBAL_EMBEDDINGS_PATH, mmap_mode='r')

N_ITEMS = global_embeddings.shape[0] - 1
SEMANTIC_DIM = global_embeddings.shape[1]
ITEM_DIM = model.semantic_head[0].out_features

print('=' * 100)
print('전체 카탈로그 평가 | FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'Device: {DEVICE}')
print(f'Best checkpoint epoch: {checkpoint["epoch"]}')
print(f'Catalog size: {N_ITEMS:,}')
print(f'Semantic dimension: {SEMANTIC_DIM}')
print(f'Model item dimension: {ITEM_DIM}')
print(f'Test sessions: {len(test_dataset):,}')
print('=' * 100)

full_test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='Building catalog vectors'
    ):
        end = min(start + CATALOG_CHUNK_SIZE, N_ITEMS + 1)

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(semantic_chunk).to(
            DEVICE,
            non_blocking=True
        )

        catalog_vectors[start:end] = model.project_items(
            semantic_chunk
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

sample_results = []

with torch.inference_mode():
    progress = tqdm(
        full_test_loader,
        desc='Full-catalog test'
    )

    for batch_idx, (sequences, locales, targets) in enumerate(progress):
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(DEVICE, non_blocking=True)

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = model(
            semantic_sequences,
            locales,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_scores, top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        )

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10 = torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        )

        ndcg_20 = torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        )

        mrr_20 = torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        )

        ndcg_10_sum += ndcg_10.sum().item()
        ndcg_20_sum += ndcg_20.sum().item()
        mrr_20_sum += mrr_20.sum().item()

        batch_size = len(targets)
        total_samples += batch_size

        if len(sample_results) < 100:
            remaining = 100 - len(sample_results)
            take = min(remaining, batch_size)

            for i in range(take):
                sample_results.append({
                    'target': int(targets[i]),
                    'rank': int(ranks[i].cpu()),
                    'top_10': top_ids[i, :10].cpu().tolist()
                })

        current_recall_20 = recall_20_sum / total_samples

        progress.set_postfix(
            recall20=f'{current_recall_20:.4f}'
        )

        del semantic_sequences
        del session_vectors
        del scores
        del top_scores
        del top_indices
        del top_ids
        del matches
        del ranks

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

metrics = {
    'model': 'semantic_sasrec_baseline',
    'checkpoint_epoch': int(checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=4)

pd.DataFrame(sample_results).to_pickle(
    EVALUATION_DIR / 'semantic_sasrec_example_predictions.pkl'
)

print()
print('=' * 100)
print('전체 카탈로그 테스트 완료 | FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Test sessions : {total_samples:,}')
print(f'Catalog items : {N_ITEMS:,}')
print()
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print()
print(f'Metrics saved to: {METRICS_PATH}')
print('=' * 100)

In [ ]:
class SemanticGRURec(nn.Module):
    def __init__(self, semantic_dim=1024, d_model=128, locale_dim=6, n_layers=2, dropout=0.1):
        super().__init__()

        self.semantic_head = nn.Sequential(
            nn.Linear(semantic_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.locale_head = nn.Linear(locale_dim, d_model)

        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )

        self.output_norm = nn.LayerNorm(d_model)

    def project_items(self, semantic_vectors):
        item_vectors = self.semantic_head(semantic_vectors)
        return F.normalize(item_vectors, dim=-1)

    def forward(self, semantic_sequences, locales, padding_mask):
        x = self.semantic_head(semantic_sequences)

        locale_vectors = self.locale_head(locales).unsqueeze(1)
        x = x + locale_vectors

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        outputs, hidden = self.gru(x)

        lengths = (~padding_mask).sum(dim=1)
        last_indexes = lengths - 1
        batch_indexes = torch.arange(x.size(0), device=x.device)

        session_vectors = outputs[
            batch_indexes,
            last_indexes
        ]

        session_vectors = self.output_norm(
            session_vectors
        )

        return F.normalize(
            session_vectors,
            dim=-1
        )

In [ ]:
def train_gru_resume(model, experiment_name='semantic_gru_baseline', epochs=3, learning_rate=1e-3, weight_decay=1e-4):
    checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    last_path = checkpoint_dir / f'{experiment_name}_last.pt'
    best_path = checkpoint_dir / f'{experiment_name}_best.pt'

    run_dir = PROJECT_ROOT / 'runs' / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    start_epoch = 1
    best_val_loss = float('inf')
    global_step = 0

    if last_path.exists():
        checkpoint = torch.load(
            last_path,
            map_location=DEVICE,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint['model_state_dict']
        )

        optimizer.load_state_dict(
            checkpoint['optimizer_state_dict']
        )

        start_epoch = checkpoint['epoch'] + 1

        best_val_loss = checkpoint.get(
            'best_val_loss',
            checkpoint['val_loss']
        )

        global_step = checkpoint.get(
            'global_step',
            checkpoint['epoch'] * len(train_loader)
        )

        print('=' * 90)
        print('체크포인트 발견 | CHECKPOINT FOUND')
        print(f'Completed epoch: {checkpoint["epoch"]}')
        print(f'Resuming from epoch: {start_epoch}')
        print(f'Previous train loss: {checkpoint["train_loss"]:.4f}')
        print(f'Previous validation loss: {checkpoint["val_loss"]:.4f}')
        print('=' * 90)

    else:
        print('No checkpoint found. Training from epoch 1.')

    writer = SummaryWriter(
        log_dir=str(run_dir),
        purge_step=global_step + 1
    )

    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()

            running_loss = 0.0
            total_batches = 0

            progress = tqdm(
                train_loader,
                desc=f'{experiment_name} | Epoch {epoch}/{epochs}'
            )

            for sequences, locales, targets in progress:
                optimizer.zero_grad(
                    set_to_none=True
                )

                logits, labels = calculate_logits(
                    model,
                    sequences,
                    locales,
                    targets
                )

                loss = criterion(
                    logits,
                    labels
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

                running_loss += loss.item()
                total_batches += 1
                global_step += 1

                average_loss = (
                    running_loss / total_batches
                )

                writer.add_scalar(
                    'Train/Loss_step',
                    loss.item(),
                    global_step
                )

                progress.set_postfix(
                    loss=f'{loss.item():.4f}',
                    avg=f'{average_loss:.4f}'
                )

            train_loss = (
                running_loss / total_batches
            )

            model.eval()

            val_loss_sum = 0.0
            val_samples = 0
            recall_10_sum = 0.0
            ndcg_10_sum = 0.0

            with torch.inference_mode():
                val_progress = tqdm(
                    val_loader,
                    total=min(
                        len(val_loader),
                        VAL_MAX_BATCHES
                    ),
                    desc='Validation',
                    leave=False
                )

                for batch_idx, (sequences, locales, targets) in enumerate(val_progress):
                    if batch_idx >= VAL_MAX_BATCHES:
                        break

                    logits, labels = calculate_logits(
                        model,
                        sequences,
                        locales,
                        targets
                    )

                    loss = criterion(
                        logits,
                        labels
                    )

                    batch_size = logits.size(0)

                    val_loss_sum += (
                        loss.item() * batch_size
                    )

                    val_samples += batch_size

                    ranking = torch.argsort(
                        logits,
                        dim=1,
                        descending=True
                    )

                    positive_ranks = (
                        (ranking == 0)
                        .nonzero(as_tuple=False)[:, 1]
                        + 1
                    )

                    recall_10_sum += (
                        positive_ranks <= 10
                    ).float().sum().item()

                    ndcg = torch.where(
                        positive_ranks <= 10,
                        1.0 / torch.log2(
                            positive_ranks.float() + 1.0
                        ),
                        torch.zeros_like(
                            positive_ranks,
                            dtype=torch.float32
                        )
                    )

                    ndcg_10_sum += (
                        ndcg.sum().item()
                    )

            val_loss = (
                val_loss_sum / val_samples
            )

            recall_10 = (
                recall_10_sum / val_samples
            )

            ndcg_10 = (
                ndcg_10_sum / val_samples
            )

            writer.add_scalar(
                'Train/Loss_epoch',
                train_loss,
                epoch
            )

            writer.add_scalar(
                'Validation/Loss',
                val_loss,
                epoch
            )

            writer.add_scalar(
                'Validation/Sampled_Recall@10',
                recall_10,
                epoch
            )

            writer.add_scalar(
                'Validation/Sampled_NDCG@10',
                ndcg_10,
                epoch
            )

            is_best = (
                val_loss < best_val_loss
            )

            if is_best:
                best_val_loss = val_loss

            checkpoint = {
                'experiment_name': experiment_name,
                'epoch': epoch,
                'global_step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'recall_10': recall_10,
                'ndcg_10': ndcg_10,
                'best_val_loss': best_val_loss
            }

            torch.save(
                checkpoint,
                last_path
            )

            if is_best:
                torch.save(
                    checkpoint,
                    best_path
                )

            print()
            print('=' * 90)
            print(f'Experiment: {experiment_name}')
            print(f'Epoch {epoch}/{epochs}')
            print(f'Train loss: {train_loss:.4f}')
            print(f'Validation loss: {val_loss:.4f}')
            print(f'Sampled Recall@10: {recall_10:.4f}')
            print(f'Sampled NDCG@10: {ndcg_10:.4f}')

            if is_best:
                print('Best checkpoint saved.')

            print('=' * 90)
            print()

    finally:
        writer.close()

    print(
        f'{experiment_name} training completed.'
    )

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gru_model = SemanticGRURec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

print(gru_model)

In [ ]:
sasrec_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

gru_parameters = sum(
    p.numel()
    for p in gru_model.parameters()
    if p.requires_grad
)

print(f'SASRec trainable parameters: {sasrec_parameters:,}')
print(f'GRU trainable parameters: {gru_parameters:,}')

In [ ]:
train_gru_resume(
    model=gru_model,
    experiment_name='semantic_gru_baseline',
    epochs=3,
    learning_rate=1e-3,
    weight_decay=1e-4
)

In [ ]:
GRU_BEST_PATH = PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'semantic_gru_baseline_best.pt'

gru_best = torch.load(
    GRU_BEST_PATH,
    map_location='cpu',
    weights_only=False
)

print(f"Best epoch: {gru_best['epoch']}")
print(f"Train loss: {gru_best['train_loss']:.4f}")
print(f"Validation loss: {gru_best['val_loss']:.4f}")
print(f"Sampled Recall@10: {gru_best['recall_10']:.4f}")
print(f"Sampled NDCG@10: {gru_best['ndcg_10']:.4f}")

In [ ]:
BEST_CHECKPOINT_PATH = GRU_BEST_PATH

METRICS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'evaluation'
    / 'semantic_gru_full_catalog_metrics.json'
)

In [ ]:
gru_model = SemanticGRURec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

GRU_BEST_PATH = PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'semantic_gru_baseline_best.pt'

checkpoint = torch.load(
    GRU_BEST_PATH,
    map_location=DEVICE,
    weights_only=False
)

gru_model.load_state_dict(
    checkpoint['model_state_dict']
)

gru_model.eval()

model = gru_model

In [ ]:
EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20_000
TOP_K = 20

EVALUATION_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

model.eval()

gc.collect()
torch.cuda.empty_cache()

N_ITEMS = global_embeddings.shape[0] - 1
SEMANTIC_DIM = global_embeddings.shape[1]
ITEM_DIM = model.semantic_head[0].out_features

test_loader_full = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print('=' * 100)
print('GRU 전체 카탈로그 평가 | GRU FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'Device: {DEVICE}')
print(f'Checkpoint epoch: {checkpoint["epoch"]}')
print(f'Catalog size: {N_ITEMS:,}')
print(f'Test sessions: {len(test_dataset):,}')
print('=' * 100)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='Building catalog vectors'
    ):
        end = min(start + CATALOG_CHUNK_SIZE, N_ITEMS + 1)

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(
            semantic_chunk
        ).to(DEVICE, non_blocking=True)

        catalog_vectors[start:end] = model.project_items(
            semantic_chunk
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

with torch.inference_mode():
    progress = tqdm(
        test_loader_full,
        desc='GRU full-catalog test'
    )

    for sequences, locales, targets in progress:
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(DEVICE, non_blocking=True)

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = model(
            semantic_sequences,
            locales,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        ).indices

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10_sum += torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        ndcg_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        mrr_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        total_samples += len(targets)

        progress.set_postfix(
            recall20=f'{recall_20_sum / total_samples:.4f}'
        )

        del semantic_sequences
        del session_vectors
        del scores
        del top_indices
        del top_ids
        del matches
        del ranks

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

metrics = {
    'model': 'semantic_gru_baseline',
    'checkpoint_epoch': int(checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(
    METRICS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(metrics, f, indent=4)

print()
print('=' * 100)
print('GRU 전체 카탈로그 테스트 완료 | GRU FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Test sessions : {total_samples:,}')
print(f'Catalog items : {N_ITEMS:,}')
print()
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print()
print(f'Metrics saved to: {METRICS_PATH}')
print('=' * 100)

del catalog_vectors
del catalog_matrix

gc.collect()
torch.cuda.empty_cache()

In [ ]:
NUMERIC_FEATURES_PATH = EMBEDDINGS_DIR / 'global_product_numeric_features.npy'
NUMERIC_COLUMNS_PATH = EMBEDDINGS_DIR / 'global_product_numeric_columns.json'

if 'num_id' in num_catalog.columns:
    NUM_ID_COLUMN = 'num_id'
elif 'remainder__num_id' in num_catalog.columns:
    NUM_ID_COLUMN = 'remainder__num_id'
else:
    raise ValueError('num_id column was not found in num_catalog')

numeric_columns = [
    column for column in num_catalog.columns
    if column != NUM_ID_COLUMN
    and pd.api.types.is_numeric_dtype(num_catalog[column])
]

numeric_grouped = (
    num_catalog
    .groupby(NUM_ID_COLUMN)[numeric_columns]
    .mean()
)

N_ITEMS = global_embeddings.shape[0] - 1
NUMERIC_DIM = len(numeric_columns)

global_numeric_features = np.zeros(
    (N_ITEMS + 1, NUMERIC_DIM),
    dtype=np.float32
)

numeric_ids = numeric_grouped.index.to_numpy(
    dtype=np.int64
)

global_numeric_features[numeric_ids] = (
    numeric_grouped.to_numpy(dtype=np.float32)
)

assert global_numeric_features.shape[0] == global_embeddings.shape[0]
assert np.all(global_numeric_features[0] == 0)
assert np.isfinite(global_numeric_features).all()

np.save(
    NUMERIC_FEATURES_PATH,
    global_numeric_features
)

with open(
    NUMERIC_COLUMNS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        numeric_columns,
        f,
        indent=4
    )

print(f'NUMERIC_DIM = {NUMERIC_DIM}')
print(f'Numeric matrix shape = {global_numeric_features.shape}')
print(f'Columns = {numeric_columns}')

In [ ]:
class SemanticNumericSASRec(nn.Module):
    def __init__(self, semantic_dim=1024, numeric_dim=7, d_model=128, locale_dim=6, max_seq_len=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()

        self.semantic_head = nn.Sequential(
            nn.Linear(semantic_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.numeric_head = nn.Sequential(
            nn.Linear(numeric_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.locale_head = nn.Linear(locale_dim, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.output_norm = nn.LayerNorm(d_model)

    def project_items(self, semantic_vectors, numeric_vectors):
        semantic_vectors = self.semantic_head(semantic_vectors)
        numeric_vectors = self.numeric_head(numeric_vectors)

        item_vectors = torch.cat(
            [semantic_vectors, numeric_vectors],
            dim=-1
        )

        item_vectors = self.fusion(item_vectors)

        return F.normalize(item_vectors, dim=-1)

    def forward(self, semantic_sequences, numeric_sequences, locales, padding_mask):
        batch_size, seq_len, _ = semantic_sequences.shape

        x = self.project_items(
            semantic_sequences,
            numeric_sequences
        )

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0).expand(
            batch_size,
            seq_len
        )

        position_vectors = self.position_embedding(
            positions
        )

        locale_vectors = self.locale_head(
            locales
        ).unsqueeze(1)

        x = x + position_vectors + locale_vectors

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                dtype=torch.bool,
                device=x.device
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        lengths = (~padding_mask).sum(dim=1)
        last_indexes = lengths - 1
        batch_indexes = torch.arange(
            batch_size,
            device=x.device
        )

        session_vectors = x[
            batch_indexes,
            last_indexes
        ]

        session_vectors = self.output_norm(
            session_vectors
        )

        return F.normalize(
            session_vectors,
            dim=-1
        )

In [ ]:
GLOBAL_EMBEDDINGS_PATH = EMBEDDINGS_DIR / 'global_product_embeddings.npy'

global_embeddings = np.load(
    GLOBAL_EMBEDDINGS_PATH,
    mmap_mode='r'
)

print(global_embeddings.shape)
print(global_embeddings.dtype)

In [ ]:
def numerical_lookup(item_ids):
    ids_numpy = item_ids.detach().cpu().numpy()

    vectors = np.asarray(
        global_numeric_features[ids_numpy],
        dtype=np.float32
    )

    return torch.from_numpy(vectors)

In [ ]:
def calculate_hybrid_logits(model, sequences, locales, targets):
    padding_mask = sequences.eq(0)

    semantic_sequences = semantic_lookup(
        sequences
    ).to(DEVICE, non_blocking=True)

    numeric_sequences = numerical_lookup(
        sequences
    ).to(DEVICE, non_blocking=True)

    target_semantics = semantic_lookup(
        targets
    ).to(DEVICE, non_blocking=True)

    target_numerics = numerical_lookup(
        targets
    ).to(DEVICE, non_blocking=True)

    negative_ids = sample_negatives(
        targets,
        N_ITEMS,
        NUM_NEGATIVES
    )

    negative_semantics = semantic_lookup(
        negative_ids
    ).to(DEVICE, non_blocking=True)

    negative_numerics = numerical_lookup(
        negative_ids
    ).to(DEVICE, non_blocking=True)

    locales = locales.to(
        DEVICE,
        non_blocking=True
    )

    padding_mask = padding_mask.to(
        DEVICE,
        non_blocking=True
    )

    session_vectors = model(
        semantic_sequences,
        numeric_sequences,
        locales,
        padding_mask
    )

    positive_vectors = model.project_items(
        target_semantics,
        target_numerics
    )

    negative_vectors = model.project_items(
        negative_semantics,
        negative_numerics
    )

    positive_scores = (
        session_vectors * positive_vectors
    ).sum(
        dim=1,
        keepdim=True
    )

    negative_scores = torch.einsum(
        'bd,bkd->bk',
        session_vectors,
        negative_vectors
    )

    logits = torch.cat(
        [positive_scores, negative_scores],
        dim=1
    )

    logits = logits / TEMPERATURE

    labels = torch.zeros(
        logits.size(0),
        dtype=torch.long,
        device=DEVICE
    )

    return logits, labels

In [ ]:
gc.collect()
torch.cuda.empty_cache()

hybrid_model = SemanticNumericSASRec(
    semantic_dim=SEMANTIC_DIM,
    numeric_dim=NUMERIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

sequences, locales, targets = next(
    iter(train_loader)
)

with torch.no_grad():
    logits, labels = calculate_hybrid_logits(
        hybrid_model,
        sequences,
        locales,
        targets
    )

print(logits.shape)
print(labels.shape)

In [ ]:
def train_hybrid_experiment(model, experiment_name='semantic_numeric_sasrec', epochs=3, learning_rate=1e-3, weight_decay=1e-4):
    checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    last_path = checkpoint_dir / f'{experiment_name}_last.pt'
    best_path = checkpoint_dir / f'{experiment_name}_best.pt'

    run_dir = PROJECT_ROOT / 'runs' / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    writer = SummaryWriter(
        log_dir=str(run_dir)
    )

    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f'{experiment_name} | Epoch {epoch}/{epochs}'
        )

        for step, (sequences, locales, targets) in enumerate(progress, start=1):
            optimizer.zero_grad(
                set_to_none=True
            )

            logits, labels = calculate_hybrid_logits(
                model,
                sequences,
                locales,
                targets
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()
            global_step += 1

            writer.add_scalar(
                'Train/Loss_step',
                loss.item(),
                global_step
            )

            progress.set_postfix(
                loss=f'{loss.item():.4f}',
                avg=f'{running_loss / step:.4f}'
            )

        train_loss = running_loss / len(train_loader)

        model.eval()

        val_loss_sum = 0.0
        val_samples = 0
        recall_10_sum = 0.0
        ndcg_10_sum = 0.0

        with torch.inference_mode():
            val_progress = tqdm(
                val_loader,
                total=min(
                    len(val_loader),
                    VAL_MAX_BATCHES
                ),
                desc='Validation',
                leave=False
            )

            for batch_idx, (sequences, locales, targets) in enumerate(val_progress):
                if batch_idx >= VAL_MAX_BATCHES:
                    break

                logits, labels = calculate_hybrid_logits(
                    model,
                    sequences,
                    locales,
                    targets
                )

                loss = criterion(
                    logits,
                    labels
                )

                batch_size = logits.size(0)

                val_loss_sum += (
                    loss.item() * batch_size
                )

                val_samples += batch_size

                ranking = torch.argsort(
                    logits,
                    dim=1,
                    descending=True
                )

                positive_ranks = (
                    (ranking == 0)
                    .nonzero(as_tuple=False)[:, 1]
                    + 1
                )

                recall_10_sum += (
                    positive_ranks <= 10
                ).float().sum().item()

                ndcg = torch.where(
                    positive_ranks <= 10,
                    1.0 / torch.log2(
                        positive_ranks.float() + 1.0
                    ),
                    torch.zeros_like(
                        positive_ranks,
                        dtype=torch.float32
                    )
                )

                ndcg_10_sum += (
                    ndcg.sum().item()
                )

        val_loss = (
            val_loss_sum / val_samples
        )

        recall_10 = (
            recall_10_sum / val_samples
        )

        ndcg_10 = (
            ndcg_10_sum / val_samples
        )

        writer.add_scalar(
            'Train/Loss_epoch',
            train_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Loss',
            val_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_Recall@10',
            recall_10,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_NDCG@10',
            ndcg_10,
            epoch
        )

        is_best = (
            val_loss < best_val_loss
        )

        if is_best:
            best_val_loss = val_loss

        checkpoint = {
            'experiment_name': experiment_name,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'recall_10': recall_10,
            'ndcg_10': ndcg_10,
            'best_val_loss': best_val_loss
        }

        torch.save(
            checkpoint,
            last_path
        )

        if is_best:
            torch.save(
                checkpoint,
                best_path
            )

        print()
        print('=' * 90)
        print(f'Experiment: {experiment_name}')
        print(f'Epoch {epoch}/{epochs}')
        print(f'Train loss: {train_loss:.4f}')
        print(f'Validation loss: {val_loss:.4f}')
        print(f'Sampled Recall@10: {recall_10:.4f}')
        print(f'Sampled NDCG@10: {ndcg_10:.4f}')

        if is_best:
            print('Best checkpoint saved.')

        print('=' * 90)
        print()

    writer.close()

    print(
        f'{experiment_name} training completed.'
    )

In [ ]:
gc.collect()
torch.cuda.empty_cache()

hybrid_model = SemanticNumericSASRec(
    semantic_dim=SEMANTIC_DIM,
    numeric_dim=NUMERIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

train_hybrid_experiment(
    model=hybrid_model,
    experiment_name='semantic_numeric_sasrec',
    epochs=3,
    learning_rate=1e-3,
    weight_decay=1e-4
)

In [ ]:
HYBRID_BEST_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'checkpoints'
    / 'semantic_numeric_sasrec_best.pt'
)

HYBRID_METRICS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'evaluation'
    / 'semantic_numeric_sasrec_full_catalog_metrics.json'
)

EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20_000
TOP_K = 20

gc.collect()
torch.cuda.empty_cache()

hybrid_model = SemanticNumericSASRec(
    semantic_dim=SEMANTIC_DIM,
    numeric_dim=NUMERIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

checkpoint = torch.load(
    HYBRID_BEST_PATH,
    map_location=DEVICE,
    weights_only=False
)

hybrid_model.load_state_dict(
    checkpoint['model_state_dict']
)

hybrid_model.eval()

N_ITEMS = global_embeddings.shape[0] - 1
ITEM_DIM = hybrid_model.semantic_head[0].out_features

test_loader_full = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print('=' * 100)
print('SEMANTIC + NUMERIC SASREC FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'Checkpoint epoch: {checkpoint["epoch"]}')
print(f'Catalog size: {N_ITEMS:,}')
print(f'Test sessions: {len(test_dataset):,}')
print('=' * 100)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='Building hybrid catalog vectors'
    ):
        end = min(
            start + CATALOG_CHUNK_SIZE,
            N_ITEMS + 1
        )

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        numeric_chunk = np.asarray(
            global_numeric_features[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(
            semantic_chunk
        ).to(DEVICE, non_blocking=True)

        numeric_chunk = torch.from_numpy(
            numeric_chunk
        ).to(DEVICE, non_blocking=True)

        catalog_vectors[start:end] = (
            hybrid_model.project_items(
                semantic_chunk,
                numeric_chunk
            )
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

with torch.inference_mode():
    progress = tqdm(
        test_loader_full,
        desc='Hybrid full-catalog test'
    )

    for sequences, locales, targets in progress:
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        numeric_sequences = np.asarray(
            global_numeric_features[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(DEVICE, non_blocking=True)

        numeric_sequences = torch.from_numpy(
            numeric_sequences
        ).to(DEVICE, non_blocking=True)

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = hybrid_model(
            semantic_sequences,
            numeric_sequences,
            locales,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        ).indices

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10_sum += torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        ndcg_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        mrr_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        total_samples += len(targets)

        progress.set_postfix(
            recall20=f'{recall_20_sum / total_samples:.4f}'
        )

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

metrics = {
    'model': 'semantic_numeric_sasrec',
    'checkpoint_epoch': int(checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(
    HYBRID_METRICS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        metrics,
        f,
        indent=4
    )

print()
print('=' * 100)
print('SEMANTIC + NUMERIC SASREC FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print('=' * 100)

del catalog_vectors
del catalog_matrix

gc.collect()
torch.cuda.empty_cache()

In [ ]:
def build_session_features(X, max_seq_len=64):
    sequence_col = 'remainder__prev_items_list'

    features = np.empty((len(X), 3), dtype=np.float32)

    for i, sequence in enumerate(tqdm(X[sequence_col], desc='Building session features')):
        sequence = sequence[-max_seq_len:]

        session_len = len(sequence)
        unique_count = len(set(sequence))
        unique_ratio = unique_count / session_len

        features[i, 0] = session_len
        features[i, 1] = unique_count
        features[i, 2] = unique_ratio

    return features


train_X_d = train.drop(columns=['target'])
train_y_d = train['target']

val_X_d = val.drop(columns=['target'])
val_y_d = val['target']

test_X_d = test.drop(columns=['target'])
test_y_d = test['target']


train_session_features_raw = build_session_features(train_X_d, MAX_SEQ_LEN)
val_session_features_raw = build_session_features(val_X_d, MAX_SEQ_LEN)
test_session_features_raw = build_session_features(test_X_d, MAX_SEQ_LEN)


SESSION_FEATURE_MEAN = train_session_features_raw.mean(axis=0)
SESSION_FEATURE_STD = train_session_features_raw.std(axis=0)

SESSION_FEATURE_STD[SESSION_FEATURE_STD == 0] = 1.0


train_session_features = (
    train_session_features_raw - SESSION_FEATURE_MEAN
) / SESSION_FEATURE_STD

val_session_features = (
    val_session_features_raw - SESSION_FEATURE_MEAN
) / SESSION_FEATURE_STD

test_session_features = (
    test_session_features_raw - SESSION_FEATURE_MEAN
) / SESSION_FEATURE_STD


print('Feature mean:', SESSION_FEATURE_MEAN)
print('Feature std:', SESSION_FEATURE_STD)
print(train_session_features.shape)

In [ ]:
class SessionFeatureDataset(Dataset):
    def __init__(self, X, y, session_features):
        self.sequence_col = 'remainder__prev_items_list'

        self.locale_cols = [
            col for col in X.columns
            if 'locale_' in col.lower()
        ]

        self.sequences = X[self.sequence_col].tolist()

        self.locales = X[
            self.locale_cols
        ].to_numpy(dtype=np.float32)

        self.session_features = np.asarray(
            session_features,
            dtype=np.float32
        )

        self.targets = y.to_numpy(
            dtype=np.int64
        )

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(
            self.sequences[idx],
            dtype=torch.long
        )

        locale = torch.tensor(
            self.locales[idx],
            dtype=torch.float32
        )

        session_features = torch.tensor(
            self.session_features[idx],
            dtype=torch.float32
        )

        target = torch.tensor(
            self.targets[idx],
            dtype=torch.long
        )

        return sequence, locale, session_features, target

In [ ]:
def collate_fn_session_features(batch, padding_value=0):
    sequences, locales, session_features, targets = zip(*batch)

    sequences = [
        sequence[-MAX_SEQ_LEN:]
        for sequence in sequences
    ]

    sequences_padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=padding_value
    )

    locales = torch.stack(
        locales,
        dim=0
    )

    session_features = torch.stack(
        session_features,
        dim=0
    )

    targets = torch.stack(
        targets,
        dim=0
    )

    return sequences_padded, locales, session_features, targets

In [ ]:
train_dataset_d = SessionFeatureDataset(
    train_X_d,
    train_y_d,
    train_session_features
)

val_dataset_d = SessionFeatureDataset(
    val_X_d,
    val_y_d,
    val_session_features
)

test_dataset_d = SessionFeatureDataset(
    test_X_d,
    test_y_d,
    test_session_features
)


train_loader_d = DataLoader(
    train_dataset_d,
    batch_size=128,
    shuffle=True,
    collate_fn=collate_fn_session_features,
    num_workers=0,
    pin_memory=True
)

val_loader_d = DataLoader(
    val_dataset_d,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn_session_features,
    num_workers=0,
    pin_memory=True
)

test_loader_d = DataLoader(
    test_dataset_d,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn_session_features,
    num_workers=0,
    pin_memory=True
)

In [ ]:
sequences, locales, session_features, targets = next(
    iter(train_loader_d)
)

print(sequences.shape)
print(locales.shape)
print(session_features.shape)
print(targets.shape)

In [ ]:
class SemanticSessionSASRec(nn.Module):
    def __init__(self, semantic_dim=1024, session_feature_dim=3, d_model=128, locale_dim=6, max_seq_len=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()

        self.semantic_head = nn.Sequential(
            nn.Linear(semantic_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.locale_head = nn.Linear(
            locale_dim,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.session_feature_head = nn.Sequential(
            nn.Linear(session_feature_dim, 32),
            nn.GELU(),
            nn.Linear(32, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.session_fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.output_norm = nn.LayerNorm(
            d_model
        )

    def project_items(self, semantic_vectors):
        item_vectors = self.semantic_head(
            semantic_vectors
        )

        return F.normalize(
            item_vectors,
            dim=-1
        )

    def forward(self, semantic_sequences, locales, session_features, padding_mask):
        batch_size, seq_len, _ = semantic_sequences.shape

        x = self.semantic_head(
            semantic_sequences
        )

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0).expand(
            batch_size,
            seq_len
        )

        position_vectors = self.position_embedding(
            positions
        )

        locale_vectors = self.locale_head(
            locales
        ).unsqueeze(1)

        x = x + position_vectors + locale_vectors

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                dtype=torch.bool,
                device=x.device
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        lengths = (~padding_mask).sum(dim=1)
        last_indexes = lengths - 1

        batch_indexes = torch.arange(
            batch_size,
            device=x.device
        )

        sequence_vector = x[
            batch_indexes,
            last_indexes
        ]

        feature_vector = self.session_feature_head(
            session_features
        )

        session_vector = torch.cat(
            [sequence_vector, feature_vector],
            dim=-1
        )

        session_vector = self.session_fusion(
            session_vector
        )

        session_vector = self.output_norm(
            session_vector
        )

        return F.normalize(
            session_vector,
            dim=-1
        )

In [ ]:
def calculate_session_logits(model, sequences, locales, session_features, targets):
    padding_mask = sequences.eq(0)

    semantic_sequences = semantic_lookup(
        sequences
    ).to(DEVICE, non_blocking=True)

    target_semantics = semantic_lookup(
        targets
    ).to(DEVICE, non_blocking=True)

    negative_ids = sample_negatives(
        targets,
        N_ITEMS,
        NUM_NEGATIVES
    )

    negative_semantics = semantic_lookup(
        negative_ids
    ).to(DEVICE, non_blocking=True)

    locales = locales.to(
        DEVICE,
        non_blocking=True
    )

    session_features = session_features.to(
        DEVICE,
        non_blocking=True
    )

    padding_mask = padding_mask.to(
        DEVICE,
        non_blocking=True
    )

    session_vectors = model(
        semantic_sequences,
        locales,
        session_features,
        padding_mask
    )

    positive_vectors = model.project_items(
        target_semantics
    )

    negative_vectors = model.project_items(
        negative_semantics
    )

    positive_scores = (
        session_vectors * positive_vectors
    ).sum(
        dim=1,
        keepdim=True
    )

    negative_scores = torch.einsum(
        'bd,bkd->bk',
        session_vectors,
        negative_vectors
    )

    logits = torch.cat(
        [positive_scores, negative_scores],
        dim=1
    )

    logits = logits / TEMPERATURE

    labels = torch.zeros(
        logits.size(0),
        dtype=torch.long,
        device=DEVICE
    )

    return logits, labels

In [ ]:
gc.collect()
torch.cuda.empty_cache()

session_model = SemanticSessionSASRec(
    semantic_dim=SEMANTIC_DIM,
    session_feature_dim=3,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

sequences, locales, session_features, targets = next(
    iter(train_loader_d)
)

with torch.no_grad():
    logits, labels = calculate_session_logits(
        session_model,
        sequences,
        locales,
        session_features,
        targets
    )

print(logits.shape)
print(labels.shape)

In [ ]:
def train_session_experiment(model, experiment_name='semantic_session_sasrec', epochs=3, learning_rate=1e-3, weight_decay=1e-4):
    checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    last_path = checkpoint_dir / f'{experiment_name}_last.pt'
    best_path = checkpoint_dir / f'{experiment_name}_best.pt'

    run_dir = PROJECT_ROOT / 'runs' / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    writer = SummaryWriter(
        log_dir=str(run_dir)
    )

    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0

        progress = tqdm(
            train_loader_d,
            desc=f'{experiment_name} | Epoch {epoch}/{epochs}'
        )

        for step, (sequences, locales, session_features, targets) in enumerate(progress, start=1):
            optimizer.zero_grad(
                set_to_none=True
            )

            logits, labels = calculate_session_logits(
                model,
                sequences,
                locales,
                session_features,
                targets
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()
            global_step += 1

            writer.add_scalar(
                'Train/Loss_step',
                loss.item(),
                global_step
            )

            progress.set_postfix(
                loss=f'{loss.item():.4f}',
                avg=f'{running_loss / step:.4f}'
            )

        train_loss = (
            running_loss / len(train_loader_d)
        )

        model.eval()

        val_loss_sum = 0.0
        val_samples = 0
        recall_10_sum = 0.0
        ndcg_10_sum = 0.0

        with torch.inference_mode():
            val_progress = tqdm(
                val_loader_d,
                total=min(
                    len(val_loader_d),
                    VAL_MAX_BATCHES
                ),
                desc='Validation',
                leave=False
            )

            for batch_idx, (sequences, locales, session_features, targets) in enumerate(val_progress):
                if batch_idx >= VAL_MAX_BATCHES:
                    break

                logits, labels = calculate_session_logits(
                    model,
                    sequences,
                    locales,
                    session_features,
                    targets
                )

                loss = criterion(
                    logits,
                    labels
                )

                batch_size = logits.size(0)

                val_loss_sum += (
                    loss.item() * batch_size
                )

                val_samples += batch_size

                ranking = torch.argsort(
                    logits,
                    dim=1,
                    descending=True
                )

                positive_ranks = (
                    (ranking == 0)
                    .nonzero(as_tuple=False)[:, 1]
                    + 1
                )

                recall_10_sum += (
                    positive_ranks <= 10
                ).float().sum().item()

                ndcg = torch.where(
                    positive_ranks <= 10,
                    1.0 / torch.log2(
                        positive_ranks.float() + 1.0
                    ),
                    torch.zeros_like(
                        positive_ranks,
                        dtype=torch.float32
                    )
                )

                ndcg_10_sum += (
                    ndcg.sum().item()
                )

        val_loss = val_loss_sum / val_samples
        recall_10 = recall_10_sum / val_samples
        ndcg_10 = ndcg_10_sum / val_samples

        writer.add_scalar(
            'Train/Loss_epoch',
            train_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Loss',
            val_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_Recall@10',
            recall_10,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_NDCG@10',
            ndcg_10,
            epoch
        )

        is_best = val_loss < best_val_loss

        if is_best:
            best_val_loss = val_loss

        checkpoint = {
            'experiment_name': experiment_name,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'recall_10': recall_10,
            'ndcg_10': ndcg_10,
            'best_val_loss': best_val_loss,
            'session_feature_mean': SESSION_FEATURE_MEAN,
            'session_feature_std': SESSION_FEATURE_STD
        }

        torch.save(
            checkpoint,
            last_path
        )

        if is_best:
            torch.save(
                checkpoint,
                best_path
            )

        print()
        print('=' * 90)
        print(f'Experiment: {experiment_name}')
        print(f'Epoch {epoch}/{epochs}')
        print(f'Train loss: {train_loss:.4f}')
        print(f'Validation loss: {val_loss:.4f}')
        print(f'Sampled Recall@10: {recall_10:.4f}')
        print(f'Sampled NDCG@10: {ndcg_10:.4f}')

        if is_best:
            print('Best checkpoint saved.')

        print('=' * 90)

    writer.close()

    print(
        f'{experiment_name} training completed.'
    )

In [ ]:
gc.collect()
torch.cuda.empty_cache()

session_model = SemanticSessionSASRec(
    semantic_dim=SEMANTIC_DIM,
    session_feature_dim=3,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

train_session_experiment(
    model=session_model,
    experiment_name='semantic_session_sasrec',
    epochs=3,
    learning_rate=1e-3,
    weight_decay=1e-4
)

In [ ]:
SESSION_BEST_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'checkpoints'
    / 'semantic_session_sasrec_best.pt'
)

SESSION_METRICS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'evaluation'
    / 'semantic_session_sasrec_full_catalog_metrics.json'
)

EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20_000
TOP_K = 20

gc.collect()
torch.cuda.empty_cache()

session_model = SemanticSessionSASRec(
    semantic_dim=SEMANTIC_DIM,
    session_feature_dim=3,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

checkpoint = torch.load(
    SESSION_BEST_PATH,
    map_location=DEVICE,
    weights_only=False
)

session_model.load_state_dict(
    checkpoint['model_state_dict']
)

session_model.eval()

N_ITEMS = global_embeddings.shape[0] - 1
ITEM_DIM = session_model.semantic_head[0].out_features

print('=' * 100)
print('SEMANTIC + SESSION FEATURES SASREC FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'Checkpoint epoch: {checkpoint["epoch"]}')
print(f'Catalog size: {N_ITEMS:,}')
print(f'Test sessions: {len(test_dataset_d):,}')
print('=' * 100)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='Building catalog vectors'
    ):
        end = min(
            start + CATALOG_CHUNK_SIZE,
            N_ITEMS + 1
        )

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(
            semantic_chunk
        ).to(
            DEVICE,
            non_blocking=True
        )

        catalog_vectors[start:end] = (
            session_model.project_items(
                semantic_chunk
            )
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

with torch.inference_mode():
    progress = tqdm(
        test_loader_d,
        desc='Session-feature full-catalog test'
    )

    for sequences, locales, session_features, targets in progress:
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(
            DEVICE,
            non_blocking=True
        )

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        session_features = session_features.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = session_model(
            semantic_sequences,
            locales,
            session_features,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        ).indices

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10_sum += torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        ndcg_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        mrr_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        total_samples += len(targets)

        progress.set_postfix(
            recall20=f'{recall_20_sum / total_samples:.4f}'
        )

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

metrics = {
    'model': 'semantic_session_sasrec',
    'checkpoint_epoch': int(checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(
    SESSION_METRICS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        metrics,
        f,
        indent=4
    )

print()
print('=' * 100)
print('SEMANTIC + SESSION FEATURES SASREC FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print()
print(f'Metrics saved to: {SESSION_METRICS_PATH}')
print('=' * 100)

del catalog_vectors
del catalog_matrix

gc.collect()
torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

deep_model = SemanticSASRec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=4,
    dropout=0.1
).to(DEVICE)

baseline_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

deep_params = sum(
    p.numel()
    for p in deep_model.parameters()
    if p.requires_grad
)

print(f'Baseline trainable parameters: {baseline_params:,}')
print(f'Deep model trainable parameters: {deep_params:,}')

In [ ]:
def train_deep_sasrec(model, epochs=3, learning_rate=1e-3, weight_decay=1e-4):
    experiment_name = 'semantic_sasrec_deep4'

    checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    run_dir = PROJECT_ROOT / 'runs' / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    last_path = checkpoint_dir / f'{experiment_name}_last.pt'
    best_path = checkpoint_dir / f'{experiment_name}_best.pt'

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    writer = SummaryWriter(
        log_dir=str(run_dir)
    )

    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f'Deep SASRec | Epoch {epoch}/{epochs}'
        )

        for step, (sequences, locales, targets) in enumerate(progress, start=1):
            optimizer.zero_grad(set_to_none=True)

            logits, labels = calculate_logits(
                model,
                sequences,
                locales,
                targets
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()
            global_step += 1

            average_loss = (
                running_loss / step
            )

            writer.add_scalar(
                'Train/Loss_step',
                loss.item(),
                global_step
            )

            progress.set_postfix(
                loss=f'{loss.item():.4f}',
                avg=f'{average_loss:.4f}'
            )

        train_loss = (
            running_loss / len(train_loader)
        )

        val_loss, recall_10, ndcg_10 = validate_deep_model(
            model,
            val_loader,
            criterion,
            max_batches=VAL_MAX_BATCHES
        )

        writer.add_scalar(
            'Train/Loss_epoch',
            train_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Loss',
            val_loss,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_Recall@10',
            recall_10,
            epoch
        )

        writer.add_scalar(
            'Validation/Sampled_NDCG@10',
            ndcg_10,
            epoch
        )

        is_best = (
            val_loss < best_val_loss
        )

        if is_best:
            best_val_loss = val_loss

        checkpoint = {
            'experiment_name': experiment_name,
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'recall_10': recall_10,
            'ndcg_10': ndcg_10,
            'best_val_loss': best_val_loss,
            'n_layers': 4,
            'd_model': 128,
            'n_heads': 4
        }

        torch.save(
            checkpoint,
            last_path
        )

        if is_best:
            torch.save(
                checkpoint,
                best_path
            )

        print()
        print('=' * 90)
        print(f'Experiment: {experiment_name}')
        print(f'Epoch {epoch}/{epochs}')
        print(f'Train loss: {train_loss:.4f}')
        print(f'Validation loss: {val_loss:.4f}')
        print(f'Sampled Recall@10: {recall_10:.4f}')
        print(f'Sampled NDCG@10: {ndcg_10:.4f}')

        if is_best:
            print('Best checkpoint saved.')

        print('=' * 90)
        print()

    writer.close()

    print('Deep SASRec training completed.')

In [ ]:
def validate_deep_model(model, val_loader, criterion, max_batches=500):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    recall_10_sum = 0.0
    ndcg_10_sum = 0.0

    with torch.inference_mode():
        progress = tqdm(
            val_loader,
            total=min(len(val_loader), max_batches),
            desc='Deep validation',
            leave=False
        )

        for batch_idx, (sequences, locales, targets) in enumerate(progress):
            if batch_idx >= max_batches:
                break

            logits, labels = calculate_logits(
                model,
                sequences,
                locales,
                targets
            )

            loss = criterion(logits, labels)

            batch_size = logits.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            ranking = torch.argsort(
                logits,
                dim=1,
                descending=True
            )

            positive_ranks = (
                (ranking == 0)
                .nonzero(as_tuple=False)[:, 1]
                + 1
            )

            recall_10_sum += (
                positive_ranks <= 10
            ).float().sum().item()

            ndcg = torch.where(
                positive_ranks <= 10,
                1.0 / torch.log2(
                    positive_ranks.float() + 1.0
                ),
                torch.zeros_like(
                    positive_ranks,
                    dtype=torch.float32
                )
            )

            ndcg_10_sum += ndcg.sum().item()

    val_loss = total_loss / total_samples
    recall_10 = recall_10_sum / total_samples
    ndcg_10 = ndcg_10_sum / total_samples

    return val_loss, recall_10, ndcg_10

In [ ]:
train_deep_sasrec(
    model=deep_model,
    epochs=3,
    learning_rate=1e-3,
    weight_decay=1e-4
)

In [ ]:
DEEP_BEST_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'checkpoints'
    / 'semantic_sasrec_deep4_best.pt'
)

DEEP_METRICS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'evaluation'
    / 'semantic_sasrec_deep4_full_catalog_metrics.json'
)

EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20_000
TOP_K = 20

gc.collect()
torch.cuda.empty_cache()

deep_model = SemanticSASRec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=4,
    dropout=0.1
).to(DEVICE)

checkpoint = torch.load(
    DEEP_BEST_PATH,
    map_location=DEVICE,
    weights_only=False
)

deep_model.load_state_dict(
    checkpoint['model_state_dict']
)

deep_model.eval()

N_ITEMS = global_embeddings.shape[0] - 1
ITEM_DIM = deep_model.semantic_head[0].out_features

test_loader_full = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print('=' * 100)
print('DEEP 4-LAYER SASREC FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'Checkpoint epoch: {checkpoint["epoch"]}')
print(f'Catalog size: {N_ITEMS:,}')
print(f'Test sessions: {len(test_dataset):,}')
print('=' * 100)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='Building deep catalog vectors'
    ):
        end = min(start + CATALOG_CHUNK_SIZE, N_ITEMS + 1)

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(
            semantic_chunk
        ).to(DEVICE, non_blocking=True)

        catalog_vectors[start:end] = deep_model.project_items(
            semantic_chunk
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

with torch.inference_mode():
    progress = tqdm(
        test_loader_full,
        desc='Deep SASRec full-catalog test'
    )

    for sequences, locales, targets in progress:
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(DEVICE, non_blocking=True)

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = deep_model(
            semantic_sequences,
            locales,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        ).indices

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10_sum += torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        ndcg_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        mrr_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        total_samples += len(targets)

        progress.set_postfix(
            recall20=f'{recall_20_sum / total_samples:.4f}'
        )

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

metrics = {
    'model': 'semantic_sasrec_deep4',
    'checkpoint_epoch': int(checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(
    DEEP_METRICS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(metrics, f, indent=4)

print()
print('=' * 100)
print('DEEP 4-LAYER SASREC FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print()
print(f'Metrics saved to: {DEEP_METRICS_PATH}')
print('=' * 100)

del catalog_vectors
del catalog_matrix

gc.collect()
torch.cuda.empty_cache()

In [ ]:
EVALUATION_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'

experiment_files = {
    'A | 2-Layer Semantic SASRec': 'semantic_sasrec_full_catalog_metrics.json',
    'B | Semantic GRU': 'semantic_gru_full_catalog_metrics.json',
    'C | Semantic + Numeric SASRec': 'semantic_numeric_sasrec_full_catalog_metrics.json',
    'D | Semantic + Session Features': 'semantic_session_sasrec_full_catalog_metrics.json',
    'E | 4-Layer Semantic SASRec': 'semantic_sasrec_deep4_full_catalog_metrics.json'
}

experiment_rows = []

for experiment_name, file_name in experiment_files.items():
    file_path = EVALUATION_DIR / file_name

    with open(file_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)

    experiment_rows.append({
        '실험 | Experiment': experiment_name,
        'Recall@1': metrics['Recall@1'],
        'Recall@5': metrics['Recall@5'],
        'Recall@10': metrics['Recall@10'],
        'Recall@20': metrics['Recall@20'],
        'NDCG@10': metrics['NDCG@10'],
        'NDCG@20': metrics['NDCG@20'],
        'MRR@20': metrics['MRR@20']
    })

experiment_results = pd.DataFrame(
    experiment_rows
)

experiment_results = experiment_results.sort_values(
    'Recall@20',
    ascending=False
).reset_index(drop=True)

experiment_results.to_csv(
    EVALUATION_DIR / 'experiment_comparison.csv',
    index=False
)

print('=' * 110)
print('실험 결과 비교 | EXPERIMENT COMPARISON')
print('=' * 110)
display(experiment_results)

In [ ]:
plot_df = experiment_results.copy()

plt.figure(figsize=(12, 6))

bars = plt.bar(
    plot_df['실험 | Experiment'],
    plot_df['Recall@20']
)

plt.title(
    '전체 카탈로그 Recall@20 비교 | Full-Catalog Recall@20 Comparison'
)

plt.xlabel(
    '실험 | Experiment'
)

plt.ylabel(
    'Recall@20'
)

plt.xticks(
    rotation=20,
    ha='right'
)

plt.grid(
    axis='y',
    alpha=0.3
)

for bar, value in zip(
    bars,
    plot_df['Recall@20']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f'{value:.4f}',
        ha='center',
        va='bottom'
    )

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'recall20_comparison.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

bars = plt.bar(
    plot_df['실험 | Experiment'],
    plot_df['NDCG@20']
)

plt.title(
    '전체 카탈로그 NDCG@20 비교 | Full-Catalog NDCG@20 Comparison'
)

plt.xlabel(
    '실험 | Experiment'
)

plt.ylabel(
    'NDCG@20'
)

plt.xticks(
    rotation=20,
    ha='right'
)

plt.grid(
    axis='y',
    alpha=0.3
)

for bar, value in zip(
    bars,
    plot_df['NDCG@20']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f'{value:.4f}',
        ha='center',
        va='bottom'
    )

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'ndcg20_comparison.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()

In [ ]:
comparison_metrics = [
    'Recall@1',
    'Recall@5',
    'Recall@10',
    'Recall@20',
    'NDCG@20',
    'MRR@20'
]

comparison_plot = experiment_results.set_index(
    '실험 | Experiment'
)[comparison_metrics].T

comparison_plot.plot(
    kind='bar',
    figsize=(14, 7)
)

plt.title(
    '모델 성능 비교 | Model Performance Comparison'
)

plt.xlabel(
    '평가 지표 | Evaluation Metric'
)

plt.ylabel(
    '점수 | Score'
)

plt.xticks(
    rotation=0
)

plt.legend(
    title='실험 | Experiment',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.grid(
    axis='y',
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'all_metrics_comparison.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()

In [ ]:
BASELINE_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'checkpoints'
    / 'semantic_sasrec_best.pt'
)

BASELINE_METRICS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'evaluation'
    / 'semantic_sasrec_full_catalog_metrics.json'
)

EVAL_BATCH_SIZE = 64
CATALOG_CHUNK_SIZE = 20_000
TOP_K = 20

gc.collect()
torch.cuda.empty_cache()

baseline_model = SemanticSASRec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=2,
    dropout=0.1
).to(DEVICE)

baseline_checkpoint = torch.load(
    BASELINE_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

baseline_model.load_state_dict(
    baseline_checkpoint['model_state_dict']
)

baseline_model.eval()

N_ITEMS = global_embeddings.shape[0] - 1
ITEM_DIM = baseline_model.semantic_head[0].out_features

baseline_test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print('=' * 100)
print('2-LAYER SEMANTIC SASREC 전체 카탈로그 평가 | 2-LAYER SEMANTIC SASREC FULL-CATALOG EVALUATION')
print('=' * 100)
print(f'체크포인트 에포크 | Checkpoint epoch: {baseline_checkpoint["epoch"]}')
print(f'카탈로그 상품 수 | Catalog size: {N_ITEMS:,}')
print(f'테스트 세션 수 | Test sessions: {len(test_dataset):,}')
print('=' * 100)

catalog_vectors = torch.empty(
    (N_ITEMS + 1, ITEM_DIM),
    dtype=torch.float32,
    device=DEVICE
)

catalog_vectors[0].zero_()

with torch.inference_mode():
    for start in tqdm(
        range(1, N_ITEMS + 1, CATALOG_CHUNK_SIZE),
        desc='카탈로그 벡터 생성 | Building catalog vectors'
    ):
        end = min(start + CATALOG_CHUNK_SIZE, N_ITEMS + 1)

        semantic_chunk = np.asarray(
            global_embeddings[start:end],
            dtype=np.float32
        )

        semantic_chunk = torch.from_numpy(
            semantic_chunk
        ).to(DEVICE, non_blocking=True)

        catalog_vectors[start:end] = baseline_model.project_items(
            semantic_chunk
        )

catalog_matrix = catalog_vectors[1:]

total_samples = 0

recall_1_sum = 0
recall_5_sum = 0
recall_10_sum = 0
recall_20_sum = 0

ndcg_10_sum = 0.0
ndcg_20_sum = 0.0
mrr_20_sum = 0.0

with torch.inference_mode():
    progress = tqdm(
        baseline_test_loader,
        desc='2-Layer SASRec 전체 카탈로그 테스트 | Full-catalog test'
    )

    for sequences, locales, targets in progress:
        padding_mask = sequences.eq(0)

        semantic_sequences = np.asarray(
            global_embeddings[sequences.numpy()],
            dtype=np.float32
        )

        semantic_sequences = torch.from_numpy(
            semantic_sequences
        ).to(DEVICE, non_blocking=True)

        locales = locales.to(
            DEVICE,
            non_blocking=True
        )

        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True
        )

        padding_mask = padding_mask.to(
            DEVICE,
            non_blocking=True
        )

        session_vectors = baseline_model(
            semantic_sequences,
            locales,
            padding_mask
        )

        scores = torch.matmul(
            session_vectors,
            catalog_matrix.T
        )

        top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1
        ).indices

        top_ids = top_indices + 1

        matches = top_ids.eq(
            targets_gpu.unsqueeze(1)
        )

        found = matches.any(dim=1)

        ranks = torch.zeros(
            len(targets_gpu),
            dtype=torch.long,
            device=DEVICE
        )

        ranks[found] = (
            matches[found]
            .float()
            .argmax(dim=1)
            + 1
        )

        recall_1_sum += ((ranks > 0) & (ranks <= 1)).sum().item()
        recall_5_sum += ((ranks > 0) & (ranks <= 5)).sum().item()
        recall_10_sum += ((ranks > 0) & (ranks <= 10)).sum().item()
        recall_20_sum += ((ranks > 0) & (ranks <= 20)).sum().item()

        ndcg_10_sum += torch.where(
            (ranks > 0) & (ranks <= 10),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        ndcg_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / torch.log2(ranks.float() + 1.0),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        mrr_20_sum += torch.where(
            (ranks > 0) & (ranks <= 20),
            1.0 / ranks.float(),
            torch.zeros_like(ranks, dtype=torch.float32)
        ).sum().item()

        total_samples += len(targets)

        progress.set_postfix(
            recall20=f'{recall_20_sum / total_samples:.4f}'
        )

recall_1 = recall_1_sum / total_samples
recall_5 = recall_5_sum / total_samples
recall_10 = recall_10_sum / total_samples
recall_20 = recall_20_sum / total_samples

ndcg_10 = ndcg_10_sum / total_samples
ndcg_20 = ndcg_20_sum / total_samples
mrr_20 = mrr_20_sum / total_samples

baseline_metrics = {
    'model': 'semantic_sasrec_baseline',
    'checkpoint_epoch': int(baseline_checkpoint['epoch']),
    'test_sessions': int(total_samples),
    'catalog_size': int(N_ITEMS),
    'Recall@1': recall_1,
    'Recall@5': recall_5,
    'Recall@10': recall_10,
    'Recall@20': recall_20,
    'NDCG@10': ndcg_10,
    'NDCG@20': ndcg_20,
    'MRR@20': mrr_20
}

with open(
    BASELINE_METRICS_PATH,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        baseline_metrics,
        f,
        indent=4
    )

print()
print('=' * 100)
print('2-LAYER SASREC 전체 카탈로그 테스트 완료 | 2-LAYER SASREC FULL-CATALOG TEST COMPLETED')
print('=' * 100)
print(f'Recall@1  : {recall_1:.6f}')
print(f'Recall@5  : {recall_5:.6f}')
print(f'Recall@10 : {recall_10:.6f}')
print(f'Recall@20 : {recall_20:.6f}')
print()
print(f'NDCG@10   : {ndcg_10:.6f}')
print(f'NDCG@20   : {ndcg_20:.6f}')
print(f'MRR@20    : {mrr_20:.6f}')
print()
print(f'평가 결과 저장 완료 | Metrics saved to: {BASELINE_METRICS_PATH}')
print('=' * 100)

del catalog_vectors
del catalog_matrix

gc.collect()
torch.cuda.empty_cache()

In [ ]:
EVALUATION_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'

experiment_files = {
    'A | 2-Layer Semantic SASRec': 'semantic_sasrec_full_catalog_metrics.json',
    'B | Semantic GRU': 'semantic_gru_full_catalog_metrics.json',
    'C | Semantic + Numeric SASRec': 'semantic_numeric_sasrec_full_catalog_metrics.json',
    'D | Semantic + Session Features': 'semantic_session_sasrec_full_catalog_metrics.json',
    'E | 4-Layer Semantic SASRec': 'semantic_sasrec_deep4_full_catalog_metrics.json'
}

rows = []

for experiment, file_name in experiment_files.items():
    path = EVALUATION_DIR / file_name

    with open(path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)

    rows.append({
        '실험 | Experiment': experiment,
        'Recall@1': metrics['Recall@1'],
        'Recall@5': metrics['Recall@5'],
        'Recall@10': metrics['Recall@10'],
        'Recall@20': metrics['Recall@20'],
        'NDCG@10': metrics['NDCG@10'],
        'NDCG@20': metrics['NDCG@20'],
        'MRR@20': metrics['MRR@20']
    })

final_results = pd.DataFrame(rows)

final_results.to_csv(
    EVALUATION_DIR / 'final_experiment_results.csv',
    index=False
)

print('=' * 110)
print('최종 실험 결과 | FINAL EXPERIMENT RESULTS')
print('=' * 110)

display(
    final_results.sort_values(
        'Recall@20',
        ascending=False
    ).reset_index(drop=True)
)

plot_df = final_results.sort_values(
    'Recall@20',
    ascending=False
)

plt.figure(figsize=(13, 6))

bars = plt.bar(
    plot_df['실험 | Experiment'],
    plot_df['Recall@20']
)

plt.title(
    '전체 카탈로그 Recall@20 비교 | Full-Catalog Recall@20 Comparison'
)

plt.xlabel(
    '실험 | Experiment'
)

plt.ylabel(
    'Recall@20'
)

plt.xticks(
    rotation=20,
    ha='right'
)

plt.grid(
    axis='y',
    alpha=0.3
)

for bar, value in zip(
    bars,
    plot_df['Recall@20']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f'{value:.4f}',
        ha='center',
        va='bottom'
    )

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'final_recall20_comparison.png',
    dpi=250,
    bbox_inches='tight'
)

plt.show()

plt.figure(figsize=(13, 6))

bars = plt.bar(
    plot_df['실험 | Experiment'],
    plot_df['NDCG@20']
)

plt.title(
    '전체 카탈로그 NDCG@20 비교 | Full-Catalog NDCG@20 Comparison'
)

plt.xlabel(
    '실험 | Experiment'
)

plt.ylabel(
    'NDCG@20'
)

plt.xticks(
    rotation=20,
    ha='right'
)

plt.grid(
    axis='y',
    alpha=0.3
)

for bar, value in zip(
    bars,
    plot_df['NDCG@20']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f'{value:.4f}',
        ha='center',
        va='bottom'
    )

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'final_ndcg20_comparison.png',
    dpi=250,
    bbox_inches='tight'
)

plt.show()

metrics_to_plot = [
    'Recall@1',
    'Recall@5',
    'Recall@10',
    'Recall@20',
    'NDCG@20',
    'MRR@20'
]

comparison = final_results.set_index(
    '실험 | Experiment'
)[metrics_to_plot].T

comparison.plot(
    kind='bar',
    figsize=(15, 7)
)

plt.title(
    '최종 모델 성능 비교 | Final Model Performance Comparison'
)

plt.xlabel(
    '평가 지표 | Evaluation Metric'
)

plt.ylabel(
    '점수 | Score'
)

plt.xticks(
    rotation=0
)

plt.legend(
    title='실험 | Experiment',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.grid(
    axis='y',
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    EVALUATION_DIR / 'final_all_metrics_comparison.png',
    dpi=250,
    bbox_inches='tight'
)

plt.show()

print('=' * 110)
print('최종 결과 저장 완료 | FINAL RESULTS SAVED')
print('=' * 110)
print(f'CSV: {EVALUATION_DIR / "final_experiment_results.csv"}')
print(f'그래프 저장 완료 | Plots saved to: {EVALUATION_DIR}')

In [ ]:
class SemanticSASRec(nn.Module):
    def __init__(self, semantic_dim=1024, d_model=128, locale_dim=6, max_seq_len=64, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()

        self.semantic_head = nn.Sequential(
            nn.Linear(semantic_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

        self.locale_head = nn.Linear(locale_dim, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.output_norm = nn.LayerNorm(d_model)

    def project_items(self, semantic_vectors):
        item_vectors = self.semantic_head(semantic_vectors)
        return F.normalize(item_vectors, dim=-1)

    def forward(self, semantic_sequences, locales, padding_mask):
        batch_size, seq_len, _ = semantic_sequences.shape

        x = self.semantic_head(semantic_sequences)

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0).expand(batch_size, seq_len)

        x = (
            x
            + self.position_embedding(positions)
            + self.locale_head(locales).unsqueeze(1)
        )

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                dtype=torch.bool,
                device=x.device
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        lengths = (~padding_mask).sum(dim=1)
        last_indexes = lengths - 1

        batch_indexes = torch.arange(
            batch_size,
            device=x.device
        )

        session_vectors = x[
            batch_indexes,
            last_indexes
        ]

        session_vectors = self.output_norm(
            session_vectors
        )

        return F.normalize(
            session_vectors,
            dim=-1
        )

In [ ]:
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

EMBEDDINGS_DIR = (
    PROJECT_ROOT
    / 'artifacts'
    / 'embeddings'
    / 'voyage4_large_1024'
)

GLOBAL_EMBEDDINGS_PATH = (
    EMBEDDINGS_DIR
    / 'global_product_embeddings.npy'
)

FINAL_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'checkpoints'
    / 'semantic_sasrec_deep4_best.pt'
)

print('=' * 100)
print('최종 모델 및 임베딩 로딩 | LOADING FINAL MODEL AND EMBEDDINGS')
print('=' * 100)

global_embeddings = np.load(
    GLOBAL_EMBEDDINGS_PATH,
    mmap_mode='r'
)

SEMANTIC_DIM = global_embeddings.shape[1]
N_ITEMS = global_embeddings.shape[0] - 1
MAX_SEQ_LEN = 64

final_model = SemanticSASRec(
    semantic_dim=SEMANTIC_DIM,
    d_model=128,
    locale_dim=6,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=4,
    n_layers=4,
    dropout=0.1
).to(DEVICE)

checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

final_model.load_state_dict(
    checkpoint['model_state_dict']
)

final_model.eval()

print(f'장치 | Device: {DEVICE}')
print(f'체크포인트 에포크 | Checkpoint epoch: {checkpoint["epoch"]}')
print(f'상품 수 | Number of items: {N_ITEMS:,}')
print(f'임베딩 행렬 | Embedding matrix: {global_embeddings.shape}')
print(f'임베딩 타입 | Embedding dtype: {global_embeddings.dtype}')
print(f'트랜스포머 레이어 수 | Transformer layers: {len(final_model.transformer.layers)}')

print('=' * 100)
print('로딩 완료 | LOADING COMPLETED')
print('=' * 100)

In [ ]:
LOOKUP_DIR = PROJECT_ROOT / 'artifacts' / 'lookup'
LOOKUP_DIR.mkdir(parents=True, exist_ok=True)

LOOKUP_PATH = LOOKUP_DIR / 'product_lookup.parquet'
ITEMS_DICT_PATH = LOOKUP_DIR / 'asin_to_num_id.pkl'

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

PRODUCTS_PATH = RAW_DIR / 'products_train.csv'
SESSIONS_PATH = RAW_DIR / 'sessions_train.csv'

EXPECTED_ITEMS = global_embeddings.shape[0] - 1

print('=' * 110)
print('상품 조회 테이블 로딩 | PRODUCT LOOKUP LOADING')
print('=' * 110)

if LOOKUP_PATH.exists():
    product_lookup = pd.read_parquet(LOOKUP_PATH)

    with open(ITEMS_DICT_PATH, 'rb') as f:
        asin_to_num_id = pickle.load(f)

    print('저장된 조회 테이블을 불러왔습니다 | Saved lookup loaded')

else:
    if ITEMS_DICT_PATH.exists():
        with open(ITEMS_DICT_PATH, 'rb') as f:
            asin_to_num_id = pickle.load(f)

        print('기존 ASIN 매핑 로드 완료 | Existing ASIN mapping loaded')

    else:
        print('행동 상품 ID 재구성 시작 | Reconstructing behavioral item IDs')

        behavioral_asins = set()

        for chunk in tqdm(
            pd.read_csv(
                SESSIONS_PATH,
                usecols=['prev_items', 'next_item'],
                chunksize=200_000
            ),
            desc='세션 상품 수집 | Collecting session items'
        ):
            behavioral_asins.update(
                chunk['next_item']
                .dropna()
                .astype(str)
                .unique()
            )

            prev_items = (
                chunk['prev_items']
                .astype(str)
                .str.findall(r"'([^']+)'")
                .explode()
                .dropna()
                .unique()
            )

            behavioral_asins.update(prev_items)

        print(
            f'행동 데이터 고유 상품 수 | Unique behavioral items: '
            f'{len(behavioral_asins):,}'
        )

        catalog_ids = pd.read_csv(
            PRODUCTS_PATH,
            usecols=['id'],
            dtype={'id': 'string'}
        )

        behavioral_order = (
            catalog_ids.loc[
                catalog_ids['id'].isin(behavioral_asins),
                'id'
            ]
            .drop_duplicates()
            .tolist()
        )

        asin_to_num_id = {
            asin: num_id
            for num_id, asin in enumerate(
                behavioral_order,
                start=1
            )
        }

        del catalog_ids
        del behavioral_asins
        del behavioral_order

        gc.collect()

        with open(ITEMS_DICT_PATH, 'wb') as f:
            pickle.dump(
                asin_to_num_id,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        print(
            'ASIN 매핑 저장 완료 | ASIN mapping saved'
        )

    if len(asin_to_num_id) != EXPECTED_ITEMS:
        raise ValueError(
            f'상품 수 불일치 | Item count mismatch: '
            f'{len(asin_to_num_id):,} != {EXPECTED_ITEMS:,}'
        )

    print('상품 메타데이터 로딩 | Loading product metadata')

    product_lookup = pd.read_csv(
        PRODUCTS_PATH,
        usecols=[
            'id',
            'locale',
            'title',
            'brand',
            'price'
        ],
        dtype={
            'id': 'string',
            'locale': 'category',
            'title': 'string',
            'brand': 'string',
            'price': 'float32'
        }
    )

    product_lookup['num_id'] = (
        product_lookup['id']
        .map(asin_to_num_id)
    )

    product_lookup = (
        product_lookup
        .dropna(subset=['num_id'])
        .copy()
    )

    product_lookup['num_id'] = (
        product_lookup['num_id']
        .astype(np.int32)
    )

    product_lookup = product_lookup[
        [
            'num_id',
            'id',
            'locale',
            'title',
            'brand',
            'price'
        ]
    ]

    product_lookup.to_parquet(
        LOOKUP_PATH,
        index=False
    )

    print(
        '상품 조회 테이블 저장 완료 | Product lookup saved'
    )

print('=' * 110)
print('조회 테이블 검증 | LOOKUP VALIDATION')
print('=' * 110)

print(
    f'조회 행 수 | Lookup rows: '
    f'{len(product_lookup):,}'
)

print(
    f'고유 상품 수 | Unique items: '
    f'{product_lookup["num_id"].nunique():,}'
)

print(
    f'ASIN 매핑 수 | ASIN mappings: '
    f'{len(asin_to_num_id):,}'
)

print(
    f'num_id 범위 | num_id range: '
    f'{product_lookup["num_id"].min():,} ... '
    f'{product_lookup["num_id"].max():,}'
)

assert len(asin_to_num_id) == EXPECTED_ITEMS
assert product_lookup['num_id'].nunique() == EXPECTED_ITEMS
assert product_lookup['num_id'].min() == 1
assert product_lookup['num_id'].max() == EXPECTED_ITEMS

print('=' * 110)
print('상품 조회 테이블 준비 완료 | PRODUCT LOOKUP READY')
print('=' * 110)

display(
    product_lookup.head()
)

In [ ]:
product_lookup = pd.read_parquet(
    PROJECT_ROOT / 'artifacts' / 'lookup' / 'product_lookup.parquet'
)

with open(
    PROJECT_ROOT / 'artifacts' / 'lookup' / 'asin_to_num_id.pkl',
    'rb'
) as f:
    asin_to_num_id = pickle.load(f)

In [ ]:
FINAL_VECTORS_PATH = (
    PROJECT_ROOT
    / 'artifacts'
    / 'embeddings'
    / 'voyage4_large_1024'
    / 'deep4_catalog_vectors.npy'
)

FINAL_DIM = 128
PROJECTION_BATCH_SIZE = 20_000
SEARCH_CHUNK_SIZE = 50_000

N_ITEMS = global_embeddings.shape[0] - 1

locale_order = sorted(
    product_lookup['locale']
    .astype(str)
    .dropna()
    .unique()
    .tolist()
)

if len(locale_order) != 6:
    raise ValueError(
        f'로케일 수 오류 | Invalid number of locales: {locale_order}'
    )

locale_to_index = {
    locale: i
    for i, locale in enumerate(locale_order)
}

print('=' * 110)
print('최종 추천 시스템 준비 | FINAL RECOMMENDATION SYSTEM INITIALIZATION')
print('=' * 110)
print(f'사용 로케일 | Locale order: {locale_order}')
print(f'전체 상품 수 | Total items: {N_ITEMS:,}')

if not FINAL_VECTORS_PATH.exists():
    print(
        '최종 상품 벡터가 없습니다. 한 번만 생성합니다 | '
        'Final item vectors not found. Creating once.'
    )

    final_catalog_vectors = np.lib.format.open_memmap(
        FINAL_VECTORS_PATH,
        mode='w+',
        dtype=np.float16,
        shape=(N_ITEMS + 1, FINAL_DIM)
    )

    final_catalog_vectors[0] = 0.0

    final_model.eval()

    with torch.inference_mode():
        for start in tqdm(
            range(1, N_ITEMS + 1, PROJECTION_BATCH_SIZE),
            desc='상품 벡터 생성 | Building item vectors'
        ):
            end = min(
                start + PROJECTION_BATCH_SIZE,
                N_ITEMS + 1
            )

            semantic_chunk = np.asarray(
                global_embeddings[start:end],
                dtype=np.float32
            )

            semantic_chunk = torch.from_numpy(
                semantic_chunk
            ).to(
                DEVICE,
                non_blocking=True
            )

            projected = final_model.project_items(
                semantic_chunk
            )

            final_catalog_vectors[start:end] = (
                projected
                .cpu()
                .numpy()
                .astype(np.float16)
            )

            del semantic_chunk
            del projected

    final_catalog_vectors.flush()
    del final_catalog_vectors

    gc.collect()
    torch.cuda.empty_cache()

    print(
        '최종 상품 벡터 저장 완료 | '
        'Final item vectors saved'
    )

final_catalog_vectors = np.load(
    FINAL_VECTORS_PATH,
    mmap_mode='r'
)

print(
    f'최종 상품 벡터 | Final item vectors: '
    f'{final_catalog_vectors.shape}'
)

print('=' * 110)


def build_locale_vector(locale):
    locale = str(locale).upper()

    if locale not in locale_to_index:
        raise ValueError(
            f'지원되지 않는 로케일 | Unsupported locale: {locale}'
        )

    vector = torch.zeros(
        (1, len(locale_order)),
        dtype=torch.float32,
        device=DEVICE
    )

    vector[
        0,
        locale_to_index[locale]
    ] = 1.0

    return vector


def get_product_metadata(num_ids, locale=None):
    num_ids = [
        int(x)
        for x in num_ids
    ]

    subset = product_lookup[
        product_lookup['num_id'].isin(num_ids)
    ].copy()

    rows = []

    for num_id in num_ids:
        item_rows = subset[
            subset['num_id'] == num_id
        ]

        selected = None

        if locale is not None:
            locale_rows = item_rows[
                item_rows['locale'].astype(str) == str(locale)
            ]

            if len(locale_rows) > 0:
                selected = locale_rows.iloc[0]

        if selected is None and len(item_rows) > 0:
            selected = item_rows.iloc[0]

        if selected is None:
            rows.append({
                'num_id': num_id,
                'id': None,
                'locale': None,
                'title': None,
                'brand': None,
                'price': None
            })

        else:
            rows.append({
                'num_id': num_id,
                'id': selected.get('id'),
                'locale': selected.get('locale'),
                'title': selected.get('title'),
                'brand': selected.get('brand'),
                'price': selected.get('price')
            })

    return pd.DataFrame(rows)


def recommend_next_items(
    session_asins,
    locale,
    top_k=10,
    exclude_seen=True
):
    final_model.eval()

    locale = str(locale).upper()

    missing_asins = [
        asin
        for asin in session_asins
        if asin not in asin_to_num_id
    ]

    session_num_ids = [
        int(asin_to_num_id[asin])
        for asin in session_asins
        if asin in asin_to_num_id
    ]

    if len(session_num_ids) == 0:
        raise ValueError(
            '유효한 상품이 세션에 없습니다 | '
            'No valid products found in session'
        )

    session_num_ids = session_num_ids[
        -MAX_SEQ_LEN:
    ]

    sequence_array = np.array(
        session_num_ids,
        dtype=np.int64
    )

    semantic_sequence = np.asarray(
        global_embeddings[sequence_array],
        dtype=np.float32
    )

    semantic_sequence = torch.from_numpy(
        semantic_sequence
    ).unsqueeze(0).to(
        DEVICE,
        non_blocking=True
    )

    locale_vector = build_locale_vector(
        locale
    )

    padding_mask = torch.zeros(
        (1, len(session_num_ids)),
        dtype=torch.bool,
        device=DEVICE
    )

    with torch.inference_mode():
        session_vector = final_model(
            semantic_sequence,
            locale_vector,
            padding_mask
        )

    best_scores = torch.empty(
        0,
        dtype=torch.float32,
        device=DEVICE
    )

    best_ids = torch.empty(
        0,
        dtype=torch.long,
        device=DEVICE
    )

    seen_ids = set(
        session_num_ids
    )

    with torch.inference_mode():
        for start in range(
            1,
            N_ITEMS + 1,
            SEARCH_CHUNK_SIZE
        ):
            end = min(
                start + SEARCH_CHUNK_SIZE,
                N_ITEMS + 1
            )

            candidate_vectors = np.asarray(
                final_catalog_vectors[start:end],
                dtype=np.float32
            )

            candidate_vectors = torch.from_numpy(
                candidate_vectors
            ).to(
                DEVICE,
                non_blocking=True
            )

            scores = torch.matmul(
                session_vector,
                candidate_vectors.T
            ).squeeze(0)

            if exclude_seen:
                for seen_id in seen_ids:
                    if start <= seen_id < end:
                        scores[
                            seen_id - start
                        ] = -torch.inf

            chunk_k = min(
                top_k,
                len(scores)
            )

            chunk_scores, chunk_positions = torch.topk(
                scores,
                k=chunk_k
            )

            chunk_ids = (
                chunk_positions
                + start
            )

            best_scores = torch.cat(
                [
                    best_scores,
                    chunk_scores
                ]
            )

            best_ids = torch.cat(
                [
                    best_ids,
                    chunk_ids
                ]
            )

            keep_k = min(
                top_k,
                len(best_scores)
            )

            best_scores, keep_positions = torch.topk(
                best_scores,
                k=keep_k
            )

            best_ids = best_ids[
                keep_positions
            ]

            del candidate_vectors
            del scores

    recommendation_ids = (
        best_ids
        .cpu()
        .numpy()
        .astype(int)
        .tolist()
    )

    recommendation_scores = (
        best_scores
        .cpu()
        .numpy()
        .tolist()
    )

    metadata = get_product_metadata(
        recommendation_ids,
        locale=locale
    )

    metadata.insert(
        0,
        '순위 | Rank',
        np.arange(
            1,
            len(metadata) + 1
        )
    )

    metadata['추천 점수 | Recommendation Score'] = (
        recommendation_scores
    )

    metadata = metadata.rename(
        columns={
            'num_id': '상품 번호 | Product ID',
            'id': 'ASIN',
            'locale': '로케일 | Locale',
            'title': '상품명 | Product Title',
            'brand': '브랜드 | Brand',
            'price': '가격 | Price'
        }
    )

    input_metadata = get_product_metadata(
        session_num_ids,
        locale=locale
    )

    input_metadata.insert(
        0,
        '순서 | Order',
        np.arange(
            1,
            len(input_metadata) + 1
        )
    )

    input_metadata = input_metadata.rename(
        columns={
            'num_id': '상품 번호 | Product ID',
            'id': 'ASIN',
            'locale': '로케일 | Locale',
            'title': '상품명 | Product Title',
            'brand': '브랜드 | Brand',
            'price': '가격 | Price'
        }
    )

    print('=' * 110)
    print('추천 생성 완료 | RECOMMENDATION COMPLETED')
    print('=' * 110)
    print(
        f'로케일 | Locale: {locale}'
    )
    print(
        f'입력 상품 수 | Input items: '
        f'{len(session_num_ids)}'
    )
    print(
        f'추천 상품 수 | Recommended items: '
        f'{len(metadata)}'
    )
    print(
        f'누락 ASIN 수 | Missing ASINs: '
        f'{len(missing_asins)}'
    )

    if len(missing_asins) > 0:
        print(
            f'누락 ASIN | Missing ASINs: '
            f'{missing_asins[:10]}'
        )

    print('=' * 110)

    return input_metadata, metadata

In [ ]:


RAW_SESSIONS_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'raw'
    / 'sessions_train.csv'
)

demo_sessions = pd.read_csv(
    RAW_SESSIONS_PATH,
    usecols=['prev_items', 'locale'],
    nrows=20_000
)

demo_sessions['prev_items_list'] = (
    demo_sessions['prev_items']
    .astype(str)
    .str.findall(r"'([^']+)'")
)

demo_sessions = demo_sessions[
    demo_sessions['prev_items_list'].str.len() >= 3
].reset_index(drop=True)

demo_row = demo_sessions.sample(
    n=1,
    random_state=42
).iloc[0]

session_asins = demo_row['prev_items_list']
session_locale = str(demo_row['locale'])

print('=' * 110)
print('실제 세션 선택 완료 | REAL SESSION SELECTED')
print('=' * 110)
print(f'로케일 | Locale: {session_locale}')
print(f'세션 길이 | Session length: {len(session_asins)}')
print(f'ASIN 목록 | ASIN list: {session_asins}')
print('=' * 110)

input_items, recommendations = recommend_next_items(
    session_asins=session_asins,
    locale=session_locale,
    top_k=10
)

print()
print('=' * 110)
print('입력 세션 | INPUT SESSION')
print('=' * 110)

display(input_items)

print()
print('=' * 110)
print('추천 결과 | RECOMMENDATIONS')
print('=' * 110)

display(recommendations)

In [ ]:
DEMO_DIR = PROJECT_ROOT / 'artifacts' / 'demo'
DEMO_DIR.mkdir(parents=True, exist_ok=True)

demo_sessions = pd.read_csv(
    RAW_SESSIONS_PATH,
    usecols=['prev_items', 'next_item', 'locale'],
    nrows=20_000
)

demo_sessions['prev_items_list'] = (
    demo_sessions['prev_items']
    .astype(str)
    .str.findall(r"'([^']+)'")
)

demo_sessions = demo_sessions[
    demo_sessions['prev_items_list'].str.len() >= 3
].reset_index(drop=True)


def run_demo(sample_index, top_k=10):
    row = demo_sessions.iloc[sample_index]

    session_asins = row['prev_items_list']
    target_asin = str(row['next_item'])
    locale = str(row['locale'])

    input_items, recommendations = recommend_next_items(
        session_asins=session_asins,
        locale=locale,
        top_k=top_k
    )

    if target_asin in asin_to_num_id:
        target_num_id = int(
            asin_to_num_id[target_asin]
        )

        target_meta = get_product_metadata(
            [target_num_id],
            locale=locale
        )

        target_meta = target_meta.rename(
            columns={
                'num_id': '상품 번호 | Product ID',
                'id': 'ASIN',
                'locale': '로케일 | Locale',
                'title': '상품명 | Product Title',
                'brand': '브랜드 | Brand',
                'price': '가격 | Price'
            }
        )

        recommendation_ids = (
            recommendations[
                '상품 번호 | Product ID'
            ]
            .astype(int)
            .tolist()
        )

        if target_num_id in recommendation_ids:
            target_rank = (
                recommendation_ids.index(
                    target_num_id
                )
                + 1
            )

            hit = True

        else:
            target_rank = None
            hit = False

    else:
        target_num_id = None
        target_meta = pd.DataFrame()
        target_rank = None
        hit = False

    print()
    print('=' * 110)
    print('실제 추천 데모 | REAL RECOMMENDATION DEMO')
    print('=' * 110)

    print(
        f'샘플 번호 | Sample index: '
        f'{sample_index}'
    )

    print(
        f'로케일 | Locale: '
        f'{locale}'
    )

    print(
        f'세션 길이 | Session length: '
        f'{len(session_asins)}'
    )

    print(
        f'정답 ASIN | True target ASIN: '
        f'{target_asin}'
    )

    print(
        f'Top-{top_k} 적중 여부 | '
        f'Top-{top_k} hit: {hit}'
    )

    if hit:
        print(
            f'정답 순위 | True target rank: '
            f'{target_rank}'
        )
    else:
        print(
            f'정답 순위 | True target rank: '
            f'Not in Top-{top_k}'
        )

    print('=' * 110)

    print()
    print('입력 세션 | INPUT SESSION')
    display(input_items)

    print()
    print('실제 다음 상품 | TRUE NEXT ITEM')
    display(target_meta)

    print()
    print('추천 결과 | RECOMMENDATIONS')
    display(recommendations)

    demo_summary = pd.DataFrame({
        '샘플 번호 | Sample Index': [sample_index],
        '로케일 | Locale': [locale],
        '세션 길이 | Session Length': [len(session_asins)],
        '정답 ASIN | True Target ASIN': [target_asin],
        f'Top-{top_k} 적중 | Top-{top_k} Hit': [hit],
        '정답 순위 | True Target Rank': [
            target_rank
        ]
    })

    input_items.to_csv(
        DEMO_DIR / f'demo_{sample_index}_input.csv',
        index=False
    )

    recommendations.to_csv(
        DEMO_DIR / f'demo_{sample_index}_recommendations.csv',
        index=False
    )

    target_meta.to_csv(
        DEMO_DIR / f'demo_{sample_index}_target.csv',
        index=False
    )

    demo_summary.to_csv(
        DEMO_DIR / f'demo_{sample_index}_summary.csv',
        index=False
    )

    return {
        'input': input_items,
        'target': target_meta,
        'recommendations': recommendations,
        'summary': demo_summary
    }

In [ ]:
demo_1 = run_demo(
    sample_index=42,
    top_k=10
)

demo_2 = run_demo(
    sample_index=500,
    top_k=10
)

demo_3 = run_demo(
    sample_index=1500,
    top_k=10
)